In [133]:
!pip install pandas numpy matplotlib scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [134]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Step 1: Data Cleaning

#### 1. Load Dataset

In [135]:
df = pd.read_csv('swiggy.csv')
df.head()

,id,name,city,rating,rating_count,cost,cuisine,lic_no,link,address,menu
0,567335,AB FOODS POINT,Abohar,--,Too Few Ratings,₹ 200,"Beverages,Pizzas",22122652000138,https://www.swiggy.com/restaurants/ab-foods-po...,"AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...",Menu/567335.json
1,531342,Janta Sweet House,Abohar,4.4,50+ ratings,₹ 200,"Sweets,Bakery",12117201000112,https://www.swiggy.com/restaurants/janta-sweet...,"Janta Sweet House, Bazar No.9, Circullar Road,...",Menu/531342.json
2,158203,theka coffee desi,Abohar,3.8,100+ ratings,₹ 100,Beverages,22121652000190,https://www.swiggy.com/restaurants/theka-coffe...,"theka coffee desi, sahtiya sadan road city",Menu/158203.json
3,187912,Singh Hut,Abohar,3.7,20+ ratings,₹ 250,"Fast Food,Indian",22119652000167,https://www.swiggy.com/restaurants/singh-hut-n...,"Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR",Menu/187912.json
4,543530,GRILL MASTERS,Abohar,--,Too Few Ratings,₹ 250,"Italian-American,Fast Food",12122201000053,https://www.swiggy.com/restaurants/grill-maste...,"GRILL MASTERS, ADA Heights, Abohar - Hanumanga...",Menu/543530.json


#### 2. Basic Inspection

In [136]:
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148541 entries, 0 to 148540
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            148541 non-null  int64 
 1   name          148455 non-null  object
 2   city          148541 non-null  object
 3   rating        148455 non-null  object
 4   rating_count  148455 non-null  object
 5   cost          148410 non-null  object
 6   cuisine       148442 non-null  object
 7   lic_no        148312 non-null  object
 8   link          148541 non-null  object
 9   address       148455 non-null  object
 10  menu          148541 non-null  object
dtypes: int64(1), object(10)
memory usage: 12.5+ MB
None
id                0
name             86
city              0
rating           86
rating_count     86
cost            131
cuisine          99
lic_no          229
link              0
address          86
menu              0
dtype: int64


#### 3. Remove Duplicates

In [137]:
df = df.drop_duplicates()

In [138]:
# Check Percentage of Missing Values
(df.isnull().sum() / len(df)) * 100


id              0.000000
name            0.057896
city            0.000000
rating          0.057896
rating_count    0.057896
cost            0.088191
cuisine         0.066648
lic_no          0.154166
link            0.000000
address         0.057896
menu            0.000000
dtype: float64

In [139]:
critical_columns = ['name', 'rating', 'rating_count', 'cost', 'cuisine']
df = df.dropna(subset=critical_columns)

#### 4. Clean rating

In [140]:
df['rating'] = df['rating'].replace('--', None)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

In [141]:
# Fill NaN using the median rating of the same City AND Cuisine
df['rating'] = df['rating'].fillna(df.groupby(['city', 'cuisine'])['rating'].transform('median'))

In [142]:
# For any leftovers, fill using just the City median
df['rating'] = df['rating'].fillna(df.groupby('city')['rating'].transform('median'))

In [143]:
# If a whole city has no ratings, use global median
df['rating'] = df['rating'].fillna(df['rating'].median())

In [144]:
df['rating'].isna().sum()

np.int64(0)

#### 5. Clean rating_count

In [145]:
# 1. Fill 'Too Few Ratings' and '--' string values with '0' BEFORE using .str accessor
df['rating_count'] = df['rating_count'].replace({'Too Few Ratings': '0', '--': '0'})

# 2. Convert to string explicitly, remove '+ ratings', and strip spaces safely
df['rating_count'] = df['rating_count'].astype(str).str.replace(r'\+ ratings', '', regex=True)
df['rating_count'] = df['rating_count'].str.strip()

# 3. Convert to numeric values
df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce')

# 4. Fill any leftover missing slots with 0 and convert to integers
df['rating_count'] = df['rating_count'].fillna(0).astype(int)

In [146]:
df['rating_count'].isna().sum()

np.int64(0)

####  6. Clean cost

In [147]:
# 1. Force the column to string type first to prevent NaN float errors
df['cost'] = df['cost'].astype(str)

# 2. Run your cleaning replacements
df['cost'] = df['cost'].str.replace('₹', '', regex=False)
df['cost'] = df['cost'].str.replace(',', '', regex=False)
df['cost'] = df['cost'].str.strip()

# 3. Convert to numeric values safely (your line)
df['cost'] = pd.to_numeric(df['cost'], errors='coerce')

# 4. CRITICAL STEP: Fill the 131 missing values with the median cost
# (This prevents downstream errors and keeps your data type consistent)
df['cost'] = df['cost'].fillna(df['cost'].median())

# 5. Convert to integer for cleaner analysis and Streamlit display
df['cost'] = df['cost'].astype(int)

#### 7. Clean cuisine

In [148]:
# 1. Fill the 99 missing values with a placeholder first to avoid string errors
df['cuisine'] = df['cuisine'].fillna('Unknown')

# 2. Convert to string and strip out quotes, brackets, and spaces
df['cuisine'] = df['cuisine'].astype(str)
df['cuisine'] = df['cuisine'].str.replace('"', '', regex=False)
df['cuisine'] = df['cuisine'].str.replace("'", "", regex=False)  # Also removes single quotes
df['cuisine'] = df['cuisine'].str.replace('[', '', regex=False)  # Removes opening list brackets
df['cuisine'] = df['cuisine'].str.replace(']', '', regex=False)  # Removes closing list brackets
df['cuisine'] = df['cuisine'].str.strip()

#### 8. Handle Missing Values

In [149]:
df.isna().sum()

id                0
name              0
city              0
rating            0
rating_count      0
cost              0
cuisine           0
lic_no          143
link              0
address           0
menu              0
dtype: int64

#### 9. Drop Unnecessary Columns (optional but recommended)

In [150]:
df['address']

0         AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...
1         Janta Sweet House, Bazar No.9, Circullar Road,...
2                theka coffee desi, sahtiya sadan road city
3           Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR
4         GRILL MASTERS, ADA Heights, Abohar - Hanumanga...
                                ...                        
148536    The Food Delight, 94MC+X35, New Singhania Naga...
148537    MAITRI FOODS & BEVERAGES, POLIC MITRYA SOCIETY...
148538    Cafe Bella Ciao, SHOP NO 2 NEMANI MARKET SBI S...
148539    GRILL ZILLA, SHO NO 2/6, POSTEL GROUND CHOWPAT...
148540    Lazeez kitchen, 94G3+2RR, Wadgaon, Yavatmal, M...
Name: address, Length: 148398, dtype: object

In [151]:
df = df.drop(columns=['id','lic_no', 'link', 'address', 'menu'])

#### 10. Reset Index

In [152]:
# Step 1: Clean hidden white spaces from your source column
df['city'] = df['city'].astype(str).str.strip()

# Step 2: Extract 'maincity' (the text AFTER the comma if it exists, else use the whole word)
df['maincity'] = df['city'].apply(lambda x: x.split(",")[-1].strip() if "," in x else x)

# Step 3: Extract the subregion 'city' (the text BEFORE the comma if it exists, else use the whole word)
# Example: 'Koramangala,Bangalore' -> 'city' becomes 'Koramangala', 'maincity' becomes 'Bangalore'
df['city'] = df['city'].apply(lambda x: x.split(",")[0].strip() if "," in x else x)

# Step 4: Reorder columns so they sit cleanly next to each other
cols = list(df.columns)
if 'maincity' in cols:
    cols.remove('maincity')
    city_idx = cols.index('city')
    cols.insert(city_idx, 'maincity')
    df = df[cols]


In [153]:
df = df.reset_index(drop=True)

#### 11. Save Cleaned Data

In [154]:
df.to_csv('cleaned_data.csv', index=False)